[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/08-rebuild/08_data_vis/08_5_Two_Numeric_Columns.ipynb)

# 08.5: Two Numeric Columns

Do penguins with longer flippers weigh more? A question like this asks about the relationship between two numeric columns. A **scatter plot** answers it by drawing one point per row, with one column on each axis. This notebook shows how to read one, introduces a number that summarizes it, the **correlation**, and goes through the main ways that both the chart and the number can mislead you.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## Reading a scatter plot

When there is a natural direction to the question, put the column you think does the explaining on the x axis and the one being explained on the y axis. When you read a scatter plot, look for four things:

- **Direction:** do the points rise or fall from left to right?
- **Form:** do they follow a straight line, or a curve?
- **Strength:** do they stay close to that form, or scatter widely around it?
- **Unusual points:** do any sit far away from the rest?

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

penguins = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_penguins.csv")
ax = sns.scatterplot(data=penguins, x="flipper_length_mm", y="body_mass_g")
ax.set_title("Flipper length and body mass")
ax.set_xlabel("Flipper length (mm)")
ax.set_ylabel("Body mass (grams)")
plt.show()
print(penguins["flipper_length_mm"].corr(penguins["body_mass_g"]))

The points rise from left to right along a fairly straight path, and they stay fairly close to it, so longer flippers go with heavier penguins. No point sits far off by itself. The **correlation** summarizes how closely two columns follow a straight line. It runs from -1 to 1. Values near 1 mean that they rise together, values near -1 mean that one falls as the other rises, and values near 0 mean that there is no straight-line relationship. Here `.corr()` gives about 0.87, which is a strong positive relationship. `.corr()` skips any row with a missing value in either column. We will not work through the formula.

## The number can hide the picture

In 1973, the statistician Francis Anscombe made up four small tables to show why you should draw your data. Each one has 11 rows with columns `x` and `y`.

In [ ]:
anscombe = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_anscombe.csv")
for name in ["I", "II", "III", "IV"]:
    rows = anscombe[anscombe["dataset"] == name]
    print(name, round(rows["x"].mean(), 2), round(rows["y"].mean(), 2), round(rows["x"].corr(rows["y"]), 3))

All four tables have an `x` mean of 9, a `y` mean of 7.5 and a correlation of about 0.82, so by these numbers they look the same. `sns.relplot()` draws scatter plots, and `col=` makes one chart per table, the same way it worked in 08.4.

In [ ]:
grid = sns.relplot(data=anscombe, x="x", y="y", col="dataset", col_wrap=2, height=3)
grid.set_axis_labels("x", "y")
plt.show()

The four charts look nothing alike. Only table I looks like what a correlation of 0.82 suggests. Table II is a smooth curve, table III is a nearly perfect line with one point far above it, and in table IV every `x` is the same except for one point, which produces the whole correlation by itself. So calculate the correlation after you have looked at the chart, and not in place of it.

## One unusual point

Table III and table IV were built to make this point, but it happens with real data as well. Here are the violent and property crime rates of the states and the District of Columbia in 2012.

In [ ]:
crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent_Rate", "Rates.Property.All": "Property_Rate", "Totals.Violent.All": "Violent_Total", "Totals.Property.All": "Property_Total"})
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
ax = sns.scatterplot(data=places_2012, x="Violent_Rate", y="Property_Rate")
ax.set_title("Violent and property crime rates, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("Property crimes per 100,000 people")
plt.show()

without_dc = places_2012[places_2012["State"] != "District of Columbia"]
print(places_2012["Violent_Rate"].corr(places_2012["Property_Rate"]))
print(without_dc["Violent_Rate"].corr(without_dc["Property_Rate"]))

The point far to the upper right is the District of Columbia. With it the correlation is about 0.67, and without it the correlation is about 0.53, so one of 51 points accounts for a large part of the number. D.C. is a real place with real rates, so it should not simply be deleted. The honest approach is to report both numbers and to explain why D.C. differs, as a city rather than a state.

The correlation so far is called **Pearson's r**, and it measures closeness to a straight line. **Spearman's rho** works from the ranks of the values instead of the values themselves, so it asks only whether higher in one column goes with higher in the other, and a single extreme point moves it less. You choose it with `method="spearman"`.

In [ ]:
print(places_2012["Violent_Rate"].corr(places_2012["Property_Rate"], method="spearman"))
print(without_dc["Violent_Rate"].corr(without_dc["Property_Rate"], method="spearman"))

Removing D.C. moves Spearman's rho only from about 0.58 to about 0.55, while Pearson's r moved from about 0.67 to about 0.53.

## Curves

A correlation measures closeness to a straight line, so it can understate a relationship that follows a curve. `sns.regplot()` draws a scatter plot and adds a **regression line**, the straight line that best fits the points. The shaded band shows the uncertainty about where that line belongs. Here one row is one car model.

In [ ]:
cars = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_mpg.csv")
ax = sns.regplot(data=cars, x="horsepower", y="mpg")
ax.set_title("Horsepower and fuel economy")
ax.set_xlabel("Horsepower")
ax.set_ylabel("Miles per gallon")
plt.show()
print(cars["horsepower"].corr(cars["mpg"]))
print(cars["horsepower"].corr(cars["mpg"], method="spearman"))

The points follow a curve: `mpg` drops quickly as horsepower rises from about 50 to about 100, and then more slowly. The straight line misses that shape, running below most of the points at the low end and at the high end, and above them in the middle. Pearson's r is about -0.78, and Spearman's rho, which does not depend on a straight line, is about -0.85. A line fits only if the points actually look like a line, so check that before you describe a relationship with one.

## Groups inside the data

Sometimes the pattern in the whole table is the opposite of the pattern inside each group. Are penguins with longer bills also the ones with deeper bills?

In [ ]:
ax = sns.regplot(data=penguins, x="bill_length_mm", y="bill_depth_mm")
ax.set_title("Bill length and bill depth, all penguins")
ax.set_xlabel("Bill length (mm)")
ax.set_ylabel("Bill depth (mm)")
plt.show()
print(penguins["bill_length_mm"].corr(penguins["bill_depth_mm"]))

The line slopes down and the correlation is about -0.24, which suggests that longer bills tend to be shallower. Now color the points by species with `hue=`. `sns.lmplot()` works like `sns.regplot()`, but it can draw one line per group, and like `sns.relplot()` it returns a grid.

In [ ]:
grid = sns.lmplot(data=penguins, x="bill_length_mm", y="bill_depth_mm", hue="species")
grid.set_axis_labels("Bill length (mm)", "Bill depth (mm)")
plt.show()
for name in ["Adelie", "Chinstrap", "Gentoo"]:
    rows = penguins[penguins["species"] == name]
    print(name, round(rows["bill_length_mm"].corr(rows["bill_depth_mm"]), 2))

Within each species the line slopes up, with correlations of about 0.39, 0.65 and 0.64. The overall downward slope comes from the differences among species: Gentoo penguins have long but shallow bills, and that pulls the overall pattern downward. So the answer for any one species is the opposite of the answer for all penguins together. A reversal like this is called **Simpson's paradox**. When your data contains groups, color the points by group before you draw a conclusion about the whole table.

## Too many points

A scatter plot has a limit. This table has 53940 diamonds, and when points land on top of each other, a chart cannot show how many are there. This is called **overplotting**.

In [ ]:
diamonds = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_diamonds.csv")
ax = sns.scatterplot(data=diamonds, x="carat", y="price")
ax.set_title("Carat and price, every diamond")
ax.set_xlabel("Weight (carats)")
ax.set_ylabel("Price (dollars)")
plt.show()

Everything below about 2 carats is a solid mass of color, so you cannot see where most of the diamonds are. `alpha=` makes the points partly see-through, so that darker areas have more points, and `s=` makes them smaller.

In [ ]:
ax = sns.scatterplot(data=diamonds, x="carat", y="price", alpha=0.02, s=10)
ax.set_title("Carat and price, with see-through points")
ax.set_xlabel("Weight (carats)")
ax.set_ylabel("Price (dollars)")
plt.show()

Now the crowded region shows up, along with vertical stripes at round weights such as 1, 1.5 and 2 carats, which are the same spikes you saw in notebook 08.3. Another option is a two-dimensional histogram, which divides the chart into small rectangles and colors each one by the number of points in it. `sns.histplot()` draws one if you give it both `x=` and `y=`.

In [ ]:
ax = sns.histplot(data=diamonds, x="carat", y="price", bins=60)
ax.set_title("Carat and price, as a two-dimensional histogram")
ax.set_xlabel("Weight (carats)")
ax.set_ylabel("Price (dollars)")
plt.show()

The darkest rectangles sit at the lower left, below about 1 carat and about 5000 dollars, where most of the diamonds are. The first chart made the thin scatter of expensive diamonds look just as important as that crowded region, and these last two do not.

## A relationship is not a cause

A correlation says that two columns move together. It does not say why. Here are the total numbers of violent crimes and property crimes in each place in 2012.

In [ ]:
ax = sns.scatterplot(data=places_2012, x="Violent_Total", y="Property_Total")
ax.set_title("Total violent and property crimes, 2012")
ax.set_xlabel("Number of violent crimes")
ax.set_ylabel("Number of property crimes")
plt.show()
print(places_2012["Violent_Total"].corr(places_2012["Property_Total"]))

The correlation is about 0.97, so the points are close to a straight line, with California at the far upper right. The `1e6` above the y axis means that its numbers are in millions. But this does not mean that violent crime and property crime are that closely tied. Both totals are large in a state with many people, so population is a third column that drives both. A column like this is called a **lurking variable**. When we divided by population earlier in this notebook, the correlation between the rates was much lower, about 0.67. When you see a strong correlation, ask what else could produce it before you conclude that one column causes the other.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

penguins = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_penguins.csv")
cars = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_mpg.csv")
diamonds = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_diamonds.csv")
gentoo_rows = penguins[penguins["species"] == "Gentoo"]

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")


def check_chart(number, name, test, hint, y_label=True):
    """Check a chart stored in a variable: it has labels, and test(chart) is True."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
        return
    chart = globals()[name]
    if not hasattr(chart, "get_title"):
        print(f"Problem {number}: not yet. Store what the seaborn function returns in {name}.")
    elif not chart.get_title() or not chart.get_xlabel() or (y_label and not chart.get_ylabel()):
        print(f"Problem {number}: not yet. Give the chart a title and label its axes.")
    else:
        try:
            ok = bool(test(chart))
        except (AttributeError, IndexError, TypeError, ValueError):
            ok = False
        print(f"Problem {number}: correct." if ok else f"Problem {number}: not yet. {hint}")

### Problem 1
Make a scatter plot with `weight` on the x axis and `mpg` on the y axis, using `cars`, and store it in `weight_scatter`. Give it a title and label both axes.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check_chart(1, "weight_scatter", lambda c: np.allclose(sorted(c.collections[0].get_offsets()[:, 0]), sorted(cars["weight"])), "Use sns.scatterplot() with data=cars, x=\"weight\" and y=\"mpg\".")

### Problem 2
Store the Pearson correlation between `weight` and `mpg` in a variable named `weight_corr`.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "weight_corr", cars["weight"].corr(cars["mpg"]), "Call .corr() on one column and pass it the other.")

### Problem 3
Look at your chart and your number. What is the direction of the relationship? Store your answer in a variable named `weight_direction`, as one of `"positive"`, `"negative"` or `"none"`.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "weight_direction", "negative", "Do the points rise or fall from left to right?")

### Problem 4
Make a scatter plot of `flipper_length_mm` on the x axis and `bill_depth_mm` on the y axis, using `penguins`, with the points colored by `species`. Store it in `species_scatter`. Give it a title and label both axes.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check_chart(4, "species_scatter", lambda c: {t.get_text() for t in c.get_legend().get_texts()} == {"Adelie", "Chinstrap", "Gentoo"} and c.get_xlim()[1] < 240, "Use sns.scatterplot() and add hue=\"species\".")

### Problem 5
Store the correlation between `flipper_length_mm` and `bill_depth_mm` for all penguins in `all_corr`, and for Gentoo penguins only in `gentoo_corr`. `gentoo_rows` holds the Gentoo rows.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "gentoo_corr", gentoo_rows["flipper_length_mm"].corr(gentoo_rows["bill_depth_mm"]), "Call .corr() on the columns of gentoo_rows.")

### Problem 6
Compare `all_corr` and `gentoo_corr` from Problem 5. Do they point the same way? Store your answer in a variable named `corr_sign`, as one of `"same"` or `"opposite"`.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "corr_sign", "opposite", "One is negative and the other is positive.")

### Problem 7
The cell below already has a scatter plot of `carat` and `price` from `diamonds`. Run it and look at how the points cover each other. Fix the code so that the points are see-through, with `alpha=0.02` and `s=10`. Keep the chart stored in `clear_scatter`.

In [ ]:
# Problem 7
clear_scatter = sns.scatterplot(data=diamonds, x="carat", y="price")
clear_scatter.set_title("Carat and price of diamonds")
clear_scatter.set_xlabel("Weight (carats)")
clear_scatter.set_ylabel("Price (dollars)")
plt.show()

In [ ]:
#@title Check Problem 7
check_chart(7, "clear_scatter", lambda c: c.collections[0].get_alpha() is not None and c.collections[0].get_alpha() <= 0.1, "Add alpha=0.02 and s=10 to the sns.scatterplot() call.")

### Problem 8
Across U.S. cities, the number of churches and the number of bars have a high positive correlation. Which is the most likely explanation? Store your answer in a variable named `lurking`, as one of `"churches cause bars"`, `"bars cause churches"` or `"population"`.

In [ ]:
# Problem 8

In [ ]:
#@title Check Problem 8
check(8, "lurking", "population", "Larger cities tend to have more of nearly everything.")

## What's next

You can now read a relationship between two numeric columns and test whether a single number describes it fairly. One special kind of numeric column remains: time. Time has a natural order, which makes a different kind of chart possible. Notebook 08.6 covers line plots, and when to use a bar chart instead.